# 04 — CPU Training (pilot)

Shared-engine training on **CPU**. Use for:
- local laptop (Experiment A) — supplementary
- Colab CPU runtime (Experiment B) — controlled reference

Pilot settings only: few images, few epochs. Never a full run here.

In [ ]:
import sys, time, json
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))

In [ ]:
import torch
from roadvision.config import load_config, config_hash
config = load_config(root / "configs" / "experiment.yaml")
device = torch.device("cpu")
print("device:", device)
print("config hash:", config_hash(config))
print("epochs (config):", config["training"]["epochs"],
      "| batch:", config["training"]["batch_size"])

In [ ]:
# Feasibility pilot on SYNTHETIC data — runs without the dataset
from roadvision.models.detector import build_detector, make_synthetic_batch
from roadvision.engine.train import build_optimizer, train_one_epoch

class SynthLoader:
    batch_size = 4
    def __iter__(self):
        for _ in range(5):
            yield make_synthetic_batch(batch_size=4, seed=0)

model = build_detector(config.to_dict())
opt = build_optimizer(model, config.to_dict())
t0 = time.perf_counter()
report = train_one_epoch(model, opt, SynthLoader(), device, epoch=0)
print(json.dumps(report.to_dict(), indent=2, default=str))
print("wall:", round(time.perf_counter()-t0, 2), "s")

In [ ]:
# Real-data pilot (only when dataset present AND you intend to run it)
raw = root / config["dataset"]["root"]
if raw.exists() and any(raw.iterdir()):
    from roadvision.data.dataset import load_split, seed_everything
    from roadvision.data.preprocessing import build_transforms
    from torch.utils.data import DataLoader
    seed_everything(config["seed"])
    train_ds, manifest = load_split(config["dataset"]["root"], config.to_dict(),
                                    "train", transforms=build_transforms(config.to_dict(), train=True))
    print("train images:", len(train_ds), "| manifest seed:", manifest.seed)
    # UNCOMMENT to actually train (pilot: 1 epoch, capped batches):
    # loader = DataLoader(train_ds, batch_size=4, shuffle=True,
    #                     num_workers=0, collate_fn=lambda b: tuple(zip(*b)))
    # model = build_detector(config.to_dict())
    # opt = build_optimizer(model, config.to_dict())
    # rep = train_one_epoch(model, opt, loader, device, epoch=0, max_batches=10)
    # print(rep)
else:
    print("Dataset absent — synthetic pilot above is the only runnable part.")